In [19]:
import duckdb
import pandas as pd

# Conectamos (o creamos) la base de datos local
con = duckdb.connect("lakehouse.duckdb")

con.execute("SELECT * FROM raw_customers limit 5").df()

,customer_id,first_name,last_name,email,phone,city,country,age,registration_date,loyalty_tier
0,1073.0,Andrea,Hernández,andrea.hernandez76@gmail.com,+57 3147960607,Medellin,Col.,-5.0,2024-02-22,Silver
1,1044.0,Felipe,Pérez,felipe.perez34@gmail.com,+57 3181170307,Barranquilla,Colombia,NaN,2023.09.05,GOLD
2,1078.0,Miguel,González,miguel.gonzalez59@gmail.com,+57 3102020480,Cartagena,Col.,200.0,2024/03/12,bronze
3,1076.0,Santiago,González,santiago.gonzalez19@gmail.com,None,Medellin,Colombia,NaN,2023/05/15,Gold
4,1080.0,Felipe,Hernández,felipe.hernandez58@gmail.com,+57 3236686328,Cartagena,Col.,200.0,2024/03/01,Silver


In [7]:
#Limpieza de cutomers
df = con.execute("SELECT * FROM raw_customers").df()

#Eliminamos filas nulas
df = df.dropna(how="all")

#Normalizamos separadores de fechas
df["registration_date"] = df["registration_date"].astype(str).str.replace(".", "-", regex=False).str.replace("/", "-", regex=False)
df["registration_date"] = pd.to_datetime(df["registration_date"], format="%Y-%m-%d", errors="coerce")

#Customer_id como entero
df["customer_id"] = df["customer_id"].astype("Int64")

#Duplicados en customer_id, nos quedamos con las fecha de registro mas antigua
df = df.sort_values("registration_date").drop_duplicates(subset="customer_id", keep="first")

#Estandarizamos las varibales a colombia
df["country"] = "COLOMBIA"

#Variables de texto en colunnas
df["first_name"] = df["first_name"].str.upper()
df["last_name"] = df["last_name"].str.upper()
df["city"] = df["city"].str.upper()

#Validamos edad y convertimos a entero
df.loc[(df["age"] < 0) | (df["age"] > 110), "age"] = pd.NA
df["age"] = df["age"].astype("Int64")

#Ponemos loyalti_tier en mayusculas y colocamos NO DETERMINADO los que no son validos
df["loyalty_tier"] = df["loyalty_tier"].str.upper()
tiers_validos = ["GOLD", "SILVER", "BRONZE"]
df.loc[~df["loyalty_tier"].isin(tiers_validos), "loyalty_tier"] = "NO DETERMINADO"

#Añdimos columna de auditoria con la fecha actual
df["updated_at"] = pd.Timestamp.now()

df.head()

,customer_id,first_name,last_name,email,phone,city,country,age,registration_date,loyalty_tier,updated_at
48,1008,MARÍA,VARGAS,maria.vargas9@gmail.com,+57 3249877065,BOGOTA,COLOMBIA,<NA>,2023-01-12,GOLD,2026-06-17 09:41:14.874111
9,1015,MATEO,DÍAZ,mateo.diaz32@gmail.com,+57 3128434500,MEDELLIN,COLOMBIA,<NA>,2023-01-16,GOLD,2026-06-17 09:41:14.874111
51,1039,LAURA,CASTRO,laura.castro9@gmail.com,+57 3111520596,BARRANQUILLA,COLOMBIA,<NA>,2023-01-21,SILVER,2026-06-17 09:41:14.874111
102,1071,MATEO,VARGAS,mateo.vargas49@gmail.com,+57 3212817745,BARRANQUILLA,COLOMBIA,<NA>,2023-01-26,GOLD,2026-06-17 09:41:14.874111
97,1037,JUAN,MORALES,juan.morales22@gmail.com,+57 3149961107,BOGOTA,COLOMBIA,<NA>,2023-01-28,BRONZE,2026-06-17 09:41:14.874111


In [ ]:
#Guardamos en base de datos
con.execute("CREATE OR REPLACE TABLE stage_customers AS SELECT * FROM df")

# Verificamos que se guardó bien
con.execute("SELECT * FROM stage_customers LIMIT 5").df()

,customer_id,first_name,last_name,email,phone,city,country,age,registration_date,loyalty_tier,updated_at
0,1008,MARÍA,VARGAS,maria.vargas9@gmail.com,+57 3249877065,BOGOTA,COLOMBIA,<NA>,2023-01-12,GOLD,2026-06-17 09:41:14.874111
1,1015,MATEO,DÍAZ,mateo.diaz32@gmail.com,+57 3128434500,MEDELLIN,COLOMBIA,<NA>,2023-01-16,GOLD,2026-06-17 09:41:14.874111
2,1039,LAURA,CASTRO,laura.castro9@gmail.com,+57 3111520596,BARRANQUILLA,COLOMBIA,<NA>,2023-01-21,SILVER,2026-06-17 09:41:14.874111
3,1071,MATEO,VARGAS,mateo.vargas49@gmail.com,+57 3212817745,BARRANQUILLA,COLOMBIA,<NA>,2023-01-26,GOLD,2026-06-17 09:41:14.874111
4,1037,JUAN,MORALES,juan.morales22@gmail.com,+57 3149961107,BOGOTA,COLOMBIA,<NA>,2023-01-28,BRONZE,2026-06-17 09:41:14.874111


In [21]:
#Consultamos las columnas que tiene la tabla products
con.execute("SELECT * FROM raw_products limit 8").df()

,product_id,product_name,category,price_usd,cost_usd,stock_units,supplier,active
0,2043,Juego de Mesa,Toys,841.55,579.97,NaN,Proveedor C,FALSE
1,2038,Base Maquillaje,Beauty,264.06,144.68,348.0,ProveedorA,false
2,2014,Sudadera Hoodie,Clothing,192.73,92.67,NaN,Proveedor C,0
3,2015,Pantalon Cargo,Clothing,106.76,70.09,474.0,Proveedor C,false
4,2013,Zapatos Running,Clothing,586.61,237.41,NaN,proveedor_a,true
5,2024,Colchoneta Yoga,Sports,822.91,346.31,NaN,proveedor_a,True
6,2040,Muneca Articulada,Toys,1164.71,756.04,70.0,None,1
7,2012,Vestido Floral,Clothing,215.17,91.61,13.0,proveedor_a,false


In [23]:
import re

#Limpieza de products
df_prod = con.execute("SELECT * FROM raw_products").df()

#Eliminamos filas nulas y duplicados de product_id
df_prod = df_prod.dropna(how="all")
df_prod["product_id"] = df_prod["product_id"].astype("Int64")
df_prod = df_prod.drop_duplicates(subset="product_id", keep="first")

#Category en mayusculas
df_prod["category"] = df_prod["category"].str.upper()

#price_usd, cost_usd, stock_units inválidos sean negativos o nulos
df_prod["price_usd"] = pd.to_numeric(df_prod["price_usd"], errors="coerce")
df_prod.loc[(df_prod["price_usd"] < 0) | (df_prod["price_usd"].isna()), "price_usd"] = 0

df_prod["cost_usd"] = pd.to_numeric(df_prod["cost_usd"], errors="coerce")
df_prod.loc[(df_prod["cost_usd"] < 0) | (df_prod["cost_usd"].isna()), "cost_usd"] = 0

df_prod["stock_units"] = pd.to_numeric(df_prod["stock_units"], errors="coerce")
df_prod.loc[(df_prod["stock_units"] < 0) | (df_prod["stock_units"].isna()), "stock_units"] = 0

#Normalizamos proveedor [letra] y colocamos NO DETERMINADO si no calza 
def normalizar_supplier(valor):
    if pd.isna(valor):
        return "NO DETERMINADO"
    match = re.search(r"proveedor[\s_]*([a-zA-Z])", str(valor), re.IGNORECASE)
    if match:
        return f"PROVEEDOR {match.group(1).upper()}"
    return "NO DETERMINADO"

df_prod["supplier"] = df_prod["supplier"].apply(normalizar_supplier)

#Normalizamos todas la variantes de active
def normalizar_active(valor):
    if pd.isna(valor):
        return False
    valor = str(valor).strip().lower()
    return valor in ["true", "1"]

df_prod["active"] = df_prod["active"].apply(normalizar_active)

#añdimos la columna de auditoria
df_prod["updated_at"] = pd.Timestamp.now()

df_prod.head(10)

,product_id,product_name,category,price_usd,cost_usd,stock_units,supplier,active,updated_at
0,2043,Juego de Mesa,TOYS,841.55,579.97,0.0,PROVEEDOR C,False,2026-06-17 09:59:20.912736
1,2038,Base Maquillaje,BEAUTY,264.06,144.68,348.0,PROVEEDOR A,False,2026-06-17 09:59:20.912736
2,2014,Sudadera Hoodie,CLOTHING,192.73,92.67,0.0,PROVEEDOR C,False,2026-06-17 09:59:20.912736
3,2015,Pantalon Cargo,CLOTHING,106.76,70.09,474.0,PROVEEDOR C,False,2026-06-17 09:59:20.912736
4,2013,Zapatos Running,CLOTHING,586.61,237.41,0.0,PROVEEDOR A,True,2026-06-17 09:59:20.912736
5,2024,Colchoneta Yoga,SPORTS,822.91,346.31,0.0,PROVEEDOR A,True,2026-06-17 09:59:20.912736
6,2040,Muneca Articulada,TOYS,1164.71,756.04,70.0,NO DETERMINADO,True,2026-06-17 09:59:20.912736
7,2012,Vestido Floral,CLOTHING,215.17,91.61,13.0,PROVEEDOR A,False,2026-06-17 09:59:20.912736
8,2016,Licuadora Pro,HOME & KITCHEN,344.12,212.26,0.0,PROVEEDOR A,False,2026-06-17 09:59:20.912736
9,2036,Shampoo Natural,BEAUTY,0.00,0.00,0.0,PROVEEDOR A,False,2026-06-17 09:59:20.912736


In [31]:
#Guardamos en base de datos
con.execute("CREATE OR REPLACE TABLE stage_products AS SELECT * FROM df_prod")
con.execute("SHOW TABLES").df()

# Verificamos que se guardó bien
#con.execute("SELECT * FROM stage_products LIMIT 5").df()

,name
0,raw_customers
1,raw_orders
2,raw_products
3,stage_customers
4,stage_products


In [26]:
con.execute("SELECT * FROM raw_orders limit 5").df()

,order_id,customer_id,product_id,quantity,unit_price_usd,total_amount_usd,order_date,ship_date,status,payment_method,discount_pct,credit_card_last4
0,3149.0,1064.0,2018.0,9.0,1385.04,12465.36,2023/11/15,2023/11/20,COMPLETED,paypal,5.0,8454.0
1,3286.0,1048.0,2035.0,9.0,85.43,768.87,2023/09/15,2023/09/28,pending,cash,NaN,7025.0
2,3451.0,1046.0,2004.0,3.0,52.98,158.94,2023/03/03,2023/03/17,Cancelled,Credit Card,10.0,4602.0
3,3300.0,1024.0,2019.0,4.0,480.09,1920.36,2024/04/06,2024/04/11,cancelled,debit_card,15.0,3762.0
4,3355.0,1014.0,2006.0,5.0,487.29,2436.45,2023/11/26,2023/12/07,COMPLETED,Credit Card,0.0,4310.0


In [27]:
#Limpieza de orders
df_ord= con.execute("SELECT * FROM raw_orders").df()

df_ord.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 520 entries, 0 to 519
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   order_id           515 non-null    float64
 1   customer_id        515 non-null    float64
 2   product_id         515 non-null    float64
 3   quantity           515 non-null    float64
 4   unit_price_usd     515 non-null    float64
 5   total_amount_usd   515 non-null    float64
 6   order_date         515 non-null    object 
 7   ship_date          515 non-null    object 
 8   status             515 non-null    object 
 9   payment_method     515 non-null    object 
 10  discount_pct       409 non-null    float64
 11  credit_card_last4  515 non-null    float64
dtypes: float64(8), object(4)
memory usage: 48.9+ KB


In [32]:

#Eliminamos filas nulas y duplicados
df_ord = df_ord.dropna(how="all")
df_ord = df_ord.drop_duplicates()

#Order_id sin nulos
df_ord = df_ord.dropna(subset=["order_id"])

#Todos los ids como enteros
df_ord["order_id"] = df_ord["order_id"].astype("Int64")
df_ord["customer_id"] = df_ord["customer_id"].astype("Int64")
df_ord["product_id"] = df_ord["product_id"].astype("Int64")

#Normalizamos los separadores en fechas
for col in ["order_date", "ship_date"]:
    df_ord[col] = df_ord[col].astype(str).str.replace(".", "-", regex=False).str.replace("/", "-", regex=False)
    df_ord[col] = pd.to_datetime(df_ord[col], format="%Y-%m-%d", errors="coerce")

#Precios y descuentos como numericos
df_ord["unit_price_usd"] = pd.to_numeric(df_ord["unit_price_usd"], errors="coerce")
df_ord["total_amount_usd"] = pd.to_numeric(df_ord["total_amount_usd"], errors="coerce")
df_ord["discount_pct"] = pd.to_numeric(df_ord["discount_pct"], errors="coerce")

#Validar descuentos
df_ord.loc[df_ord["discount_pct"] > 100, "discount_pct"] = pd.NA

#Quantity invalido si es negativo o nulo, recalculado de total_amount_usd/unit_price_usd
df_ord["quantity"] = pd.to_numeric(df_ord["quantity"], errors="coerce")
invalido = (df_ord["quantity"] <= 0) | (df_ord["quantity"].isna())
recalculo = df_ord["total_amount_usd"] / df_ord["unit_price_usd"]
df_ord.loc[invalido, "quantity"] = recalculo[invalido]
df_ord["quantity"] = df_ord["quantity"].round().astype("Int64")

#ship_date ANTES de order_date es imposible, verificamos
df_ord.loc[df_ord["ship_date"] < df_ord["order_date"], "ship_date"] = pd.NaT

#status mayusculas y si no existe colocamos NO DETERMINADO
df_ord["status"] = df_ord["status"].str.upper()
estados_validos = ["COMPLETED", "PENDING", "CANCELLED"]
df_ord.loc[~df_ord["status"].isin(estados_validos), "status"] = "NO DETERMINADO"

#Normalizamos payment method, guiones abajo lo ponemos como espacio y todo en mayusculas
df_ord["payment_method"] = df_ord["payment_method"].str.upper().str.replace("_", " ", regex=False)
metodos_validos = ["CASH", "CREDIT CARD", "DEBIT CARD", "PAYPAL"]
df_ord.loc[~df_ord["payment_method"].isin(metodos_validos), "payment_method"] = "NO DETERMINADO"

#customer_id y product_id deben existir en las tabals stage
clientes_validos = con.execute("SELECT customer_id FROM stage_customers").df()["customer_id"]
productos_validos = con.execute("SELECT product_id FROM stage_products").df()["product_id"]
df_ord = df_ord[df_ord["customer_id"].isin(clientes_validos) & df_ord["product_id"].isin(productos_validos)]

#Limpiar numero de tarjeta
def limpiar_tarjeta(valor):
    if pd.isna(valor):
        return None
    return str(int(valor))

df_ord["credit_card_last4"] = df_ord["credit_card_last4"].apply(limpiar_tarjeta)

#Agregamos la columna de auditoria
df_ord["updated_at"] = pd.Timestamp.now()

df_ord.head(10)

,order_id,customer_id,product_id,quantity,unit_price_usd,total_amount_usd,order_date,ship_date,status,payment_method,discount_pct,credit_card_last4,updated_at
0,3149,1064,2018,9,1385.04,12465.36,2023-11-15,2023-11-20,COMPLETED,PAYPAL,5.0,8454,2026-06-17 10:22:04.855473
1,3286,1048,2035,9,85.43,768.87,2023-09-15,2023-09-28,PENDING,CASH,NaN,7025,2026-06-17 10:22:04.855473
2,3451,1046,2004,3,52.98,158.94,2023-03-03,2023-03-17,CANCELLED,CREDIT CARD,10.0,4602,2026-06-17 10:22:04.855473
3,3300,1024,2019,4,480.09,1920.36,2024-04-06,2024-04-11,CANCELLED,DEBIT CARD,15.0,3762,2026-06-17 10:22:04.855473
4,3355,1014,2006,5,487.29,2436.45,2023-11-26,2023-12-07,COMPLETED,CREDIT CARD,0.0,4310,2026-06-17 10:22:04.855473
5,3331,1068,2031,7,1052.57,7367.99,2024-01-17,2024-01-22,PENDING,CREDIT CARD,NaN,5422,2026-06-17 10:22:04.855473
6,3145,1039,2031,5,1315.73,6578.65,2023-01-10,2023-01-12,NO DETERMINADO,CASH,20.0,9590,2026-06-17 10:22:04.855473
7,3069,1076,2014,1,443.58,443.58,2024-01-19,2024-01-28,COMPLETED,CREDIT CARD,5.0,2293,2026-06-17 10:22:04.855473
8,3013,1042,2042,6,36.45,218.70,2023-08-24,2023-08-26,COMPLETED,CREDIT CARD,10.0,2160,2026-06-17 10:22:04.855473
9,3464,1017,2026,1,440.82,440.82,2023-03-06,2023-03-10,CANCELLED,PAYPAL,20.0,6616,2026-06-17 10:22:04.855473


In [33]:
con.execute("CREATE OR REPLACE TABLE stage_orders AS SELECT * FROM df_ord")
con.execute("SELECT COUNT(*) FROM stage_orders").df()

,count_star()
0,473


In [34]:
con.execute("SHOW TABLES").df()

,name
0,raw_customers
1,raw_orders
2,raw_products
3,stage_customers
4,stage_orders
5,stage_products


In [35]:
con.close()

## Resumen - Limpieza y Transformación (RAW → STAGE)

Se aplicaron reglas de limpieza específicas a las 3 tablas, siguiendo las 
indicaciones del enunciado y las reglas adicionales de negocio:

**Customers:** estandarización de `country` a "COLOMBIA" (única opción válida), 
normalización de fechas con separadores mixtos (`-`, `.`, `/`) bajo un mismo 
formato, deduplicación de `customer_id` conservando el registro con fecha de 
registro más antigua, texto en mayúsculas, edades imposibles (negativas o 
>110) invalidadas a NULL, y `loyalty_tier` normalizado contra categorías 
válidas (GOLD, SILVER, BRONZE), con vacíos/inválidos marcados como 
"NO DETERMINADO".

**Products:** deduplicación de `product_id`, `price_usd`/`cost_usd`/
`stock_units` inválidos (negativos o nulos) reemplazados por 0, `supplier` 
normalizado al formato "PROVEEDOR [LETRA]" mediante reconocimiento de patrón 
flexible (insensible a mayúsculas, espacios o guiones bajos), y `active` 
consolidado a exactamente 2 valores booleanos.

**Orders:** IDs convertidos a enteros, eliminación de registros con `order_id` 
nulo, `quantity` inválido (≤0 o nulo) recalculado a partir de 
`total_amount_usd / unit_price_usd`, fechas normalizadas, `discount_pct` >100% 
invalidado a NULL, `status` y `payment_method` normalizados contra categorías 
válidas, `ship_date` anterior a `order_date` marcado como NULL (corrección 
lógica de negocio), integridad referencial validada contra `stage_customers` 
y `stage_products`, y `credit_card_last4` formateado como texto limpio.

En todos los casos se agregó una columna `updated_at` con la fecha y hora de 
la transformación, y el proceso es completamente re-procesable: al usar 
`CREATE OR REPLACE TABLE`, cada ejecución reconstruye las tablas STAGE desde 
la fuente más reciente, sin generar duplicados.